In [1]:
import csv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

# 1. Setup Client
client = OpenAI(api_key="YOUR_OPENAI_API_KEY")

# 2. Define the expected structure
class AnalyzedMessage(BaseModel):
    sender: str
    topic: str
    sentiment: str
    urgency_level: int  # 1-10
    key_entities: List[str]

# 3. Unstructured Input
unstructured_text = """
Subject: Urgent: Order #5521 delay. 
Hi team, I'm John from TechCorp. I'm very frustrated that my 
shipment of 50 monitors hasn't arrived in London yet. 
Please fix this immediately!
"""

# 4. Analyze using ChatGPT with Structured Output
completion = client.beta.chat.completions.parse(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": "Extract structured data from the message."},
        {"role": "user", "content": unstructured_text},
    ],
    response_format=AnalyzedMessage,
)

# 5. Extract the parsed object
result = completion.choices[0].message.parsed

# 6. Write to Structured CSV
csv_file = "analyzed_messages.csv"
fieldnames = ["sender", "topic", "sentiment", "urgency_level", "key_entities"]

with open(csv_file, mode="a", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    
    # Write header only if file is new
    file.seek(0, 2)
    if file.tell() == 0:
        writer.writeheader()
    
    # Convert list of entities to a string for CSV compatibility
    row = result.model_dump()
    row["key_entities"] = ", ".join(row["key_entities"])
    
    writer.writerow(row)

print(f"Analysis saved to {csv_file}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: YOUR_OPE*******_KEY. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}